# FactLedger extractor

`load(path) -> documents, units` over raw files and nothing else. The extractor sniffs the
format from the bytes and writes document and unit JSON in the shapes of SCHEMA.md; the
rules it follows are in BUILD.md. Built one block at a time. Inputs: the public raw dataset
and the private papers dataset, both attached to this notebook.


In [ ]:
# Block 1: inputs and integrity.
# Mount both datasets, count files per folder, and check every file's sha256 against the
# folder manifest. The manifests are used here only to prove the Kaggle copies are the bytes
# that were uploaded; the extractor itself never reads them.
import hashlib
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

def mount(slug):
    """Kaggle mounts inputs at /kaggle/input/<slug> or, in newer sessions,
    /kaggle/input/datasets/<owner>/<slug>. Take whichever exists."""
    for candidate in (Path("/kaggle/input") / slug, Path("/kaggle/input/datasets/jhffmn") / slug):
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(slug)


RAW = mount("it494-narrative-corpora-raw")
PAPERS = mount("it494-reference-papers")


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def check(folder, rows_key):
    manifest = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))
    rows = manifest[rows_key]
    on_disk = {p.name for p in folder.iterdir() if p.name not in ("manifest.json", "LICENSE")}
    listed = {r["file"] for r in rows}
    # Kaggle inputs are a network filesystem: one file at a time, 19,206 files take tens of
    # minutes; 32 concurrent reads take about a minute.
    with ThreadPoolExecutor(max_workers=32) as pool:
        digests = list(pool.map(sha256, [folder / r["file"] for r in rows]))
    bad = [r["file"] for r, d in zip(rows, digests) if d != r["sha256"]]
    print(f"{folder.name:<28} files {len(on_disk):>6}  listed {len(listed):>6}"
          f"  mismatched {len(bad)}  unlisted {len(on_disk - listed)}  missing {len(listed - on_disk)}")
    return bad


# The three literature manifests keep their original "works" key; the unpacked folders
# and the papers use "files".
for name, key in [("oz", "works"), ("holmes", "works"), ("greek", "works"),
                  ("graphrag-bench", "files"), ("longmemeval", "files")]:
    check(RAW / name, key)
check(PAPERS, "files")


In [ ]:
# Block 2: file type, then raw text.
#
# Two steps, by bytes only. Nothing here decides what the text is about; that is the model's
# job later.
#   1. file_kind(data): look at the first bytes and name the container: pdf, json, or text.
#   2. to_text(path): turn the container into one string, the document text.
#        pdf  -> the text layer, page by page (PyMuPDF, the one dependency)
#        json -> if it holds chat turns, one "role: content" block per turn under a header
#                of the session id and dates. We also keep where each turn starts and ends
#                in that string, so a chat can be cut into units without a model.
#        text -> the bytes decoded as UTF-8, unchanged
import json

try:
    import pymupdf
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf"], check=True)
    import pymupdf


def file_kind(data):
    if data.startswith(b"%PDF-"):
        return "pdf"
    if data.lstrip()[:1] in (b"{", b"["):
        return "json"
    return "text"


def pdf_text(data):
    doc = pymupdf.open(stream=data, filetype="pdf")
    return "\n".join(page.get_text() for page in doc)


def chat_turns(obj):
    """The list of {role, content} turns inside a chat JSON, wherever it sits; None if absent."""
    if isinstance(obj, list) and obj and all(isinstance(t, dict) and "role" in t and "content" in t for t in obj):
        return obj
    if isinstance(obj, dict):
        for value in obj.values():
            found = chat_turns(value)
            if found:
                return found
    return None


def chat_text(obj, turns):
    """Header lines, a blank line, then 'role: content' per turn. Returns the text and the
    (start, end) of each turn inside it."""
    header = [f"session_id: {obj['session_id']}"] if "session_id" in obj else []
    header += [f"date: {d}" for d in obj.get("dates", [])]
    text = "\n".join(header) + "\n\n"
    spans = []
    for turn in turns:
        start = len(text)
        text += f"{turn['role']}: {turn['content']}\n\n"
        spans.append((start, len(text)))
    return text, spans


def to_text(path):
    data = path.read_bytes()
    doc = {"path": str(path), "sha256": hashlib.sha256(data).hexdigest(),
           "kind": file_kind(data), "text": "", "turns": None, "dates": []}
    if doc["kind"] == "pdf":
        doc["text"] = pdf_text(data)
    elif doc["kind"] == "json":
        obj = json.loads(data)
        turns = chat_turns(obj)
        if turns is None:
            doc["kind"], doc["text"] = "text", data.decode("utf-8", errors="replace")
        else:
            doc["kind"] = "chat"
            doc["text"], doc["turns"] = chat_text(obj, turns)
            doc["dates"] = list(obj.get("dates", []))
    else:
        doc["text"] = data.decode("utf-8", errors="replace")
    return doc


# One of each, to see the shape.
for path in [RAW / "oz" / "01_55.txt", RAW / "graphrag-bench" / "Novel-30752.txt",
             RAW / "longmemeval" / "sharegpt_yywfIrx_0.json", RAW / "longmemeval" / "001cefa7_2.json",
             PAPERS / "edge2024-graphrag.pdf"]:
    d = to_text(path)
    turns = len(d["turns"]) if d["turns"] else "-"
    print(f"{path.name:<26} {d['kind']:<5} {len(d['text']):>8,} chars  turns {turns:>3}  dates {d['dates']}")
    print("    " + repr(d["text"][:70]))


In [ ]:
# Block 3: the model call, and the check that a quoted string exists in the raw text.
import json
import time

import requests
from kaggle_secrets import UserSecretsClient

MODEL = "gpt-5.6-luna"
RETRY = "gpt-5.6-terra"
PRICE = {"gpt-5.6-luna": (0.20, 1.20), "gpt-5.6-terra": (2.00, 12.00)}   # $ per M tokens in, out
SPEND_STOP = 8.00                                                        # dollars; the run halts past this
KEY = UserSecretsClient().get_secret("OPENAI_API_KEY")
calls = []


def spend():
    return sum(c["cost"] for c in calls)


def generate(prompt, model=MODEL, effort="low"):
    """One JSON-mode call; the reply parsed, the cost logged. The API rejects temperature."""
    if spend() >= SPEND_STOP:
        raise RuntimeError(f"spending stop: ${spend():.2f}")
    t0 = time.time()
    r = requests.post("https://api.openai.com/v1/chat/completions",
                      headers={"Authorization": f"Bearer {KEY}"}, timeout=300,
                      json={"model": model, "reasoning_effort": effort,
                            "response_format": {"type": "json_object"},
                            "messages": [{"role": "user", "content": prompt}]})
    if r.status_code != 200:
        raise RuntimeError(f"OpenAI {r.status_code}: {r.text}")
    body = r.json()
    u, (p_in, p_out) = body["usage"], PRICE[model]
    calls.append({"model": body["model"], "in": u["prompt_tokens"], "out": u["completion_tokens"],
                  "seconds": round(time.time() - t0, 1),
                  "cost": (u["prompt_tokens"] * p_in + u["completion_tokens"] * p_out) / 1e6})
    return json.loads(body["choices"][0]["message"]["content"])


def find(text, quote, start=0):
    """Character position of the quote at or after start, exactly as written and standing on
    its own: at least three characters, not glued to a letter on either side. None if absent."""
    if not quote or len(quote.strip()) < 3:
        return None
    at = text.find(quote, start)
    while at >= 0:
        before = text[at - 1] if at > 0 else " "
        after = text[at + len(quote)] if at + len(quote) < len(text) else " "
        if not before.isalpha() and not after.isalpha():
            return at
        at = text.find(quote, at + 1)
    return None


print(f"model {MODEL}, retry {RETRY}, spend stop ${SPEND_STOP:.2f}, key {'present' if KEY else 'MISSING'}")


In [ ]:
# Block 5: one pass of the split call. The text goes to the model in slices of about 50,000
# tokens cut at line breaks; every slice gets the same question; the answers are joined.
SLICE = 200_000      # characters per call
CAP_WORDS = 4000

PROMPT = """Below is part %d of %d of one document, as raw text. Answer with JSON only. Every string you return must be copied from the text exactly, character for character, never paraphrased or corrected, because a program will search the text for it. A heading that spans two lines (a number on one line, its title on the next) is quoted with both lines and the line break between them.

{
  "source_class": one of "canonical" (a published literary or classic work), "published" (a paper, article, or report), "authored" (a person's own material: notes, email, letters, drafts); null unless this is part 1,
  "title": the title as written, or null,
  "author": the author's name as written, or null,
  "date": {"quote": the complete line containing the date the work was written, published, or sent, "iso": "YYYY" or "YYYY-MM" or "YYYY-MM-DD"} or null. Not a transcription or ebook release date,
  "contents_end": the complete last line of the contents list, or null if this part has no contents list,
  "body_start": the complete first line of the work itself, or null if the work does not begin in this part. Publisher notices, a contents list, and transcriber's or translator's notes are not part of the work; an author's own preface or introduction is,
  "end_matter_start": the complete first line of any end matter that follows the work (references, license, index, notes, advertisements), or null if this part has no such line,
  "toc_count": the number of pieces the contents list gives, or null,
  "pieces": [{"marker": the complete heading line that begins a piece in this part, exactly as it appears where the piece begins, never as the contents list writes it, "title": a short title for the piece, such as "Chapter 1: The Cyclone" or "Abstract" or "Act II, Scene 1"}]
}

Pieces are the document's own divisions: chapters, acts and scenes, sections, dated entries, poems, stories. A paper's pieces are its sections, the abstract first. Aim for pieces under %d words; where a division is longer, use its next level down. A part with no piece beginning in it gets an empty pieces list.
%s
TEXT:
%s
"""

FIRST = ("source_class", "title", "author", "date", "contents_end", "body_start", "toc_count")   # first answer wins
LAST = ("end_matter_start",)                                                                  # last answer wins


def slices(text):
    out, start = [], 0
    while start < len(text):
        end = min(len(text), start + SLICE)
        if end < len(text):
            cut_at = text.rfind("\n", start, end)
            end = cut_at + 1 if cut_at > start else end
        out.append(text[start:end])
        start = end
    return out


def propose(doc, model=MODEL, feedback=""):
    """One merged reply for the document, from the same question asked of every slice."""
    parts = slices(doc["text"])
    reply = {"pieces": []}
    for i, part in enumerate(parts):
        r = generate(PROMPT % (i + 1, len(parts), CAP_WORDS, feedback, part), model=model)
        for key in FIRST:
            if reply.get(key) is None and r.get(key) is not None:
                reply[key] = r[key]
        for key in LAST:
            if r.get(key) is not None:
                reply[key] = r[key]
        reply["pieces"] += r.get("pieces") or []
    return reply


In [ ]:
# Block 6: from a reply to pieces.
#   locate_all   every quoted line found in the raw text, or listed as missing. Searching
#                starts after the contents list, so a heading is found where the piece begins
#                and not where the contents list names it. Headings are searched in order,
#                each from the previous find. End matter is searched from the body start.
#   until_found  the call repeated, with the missing strings fed back, until nothing is
#                missing; two tries, then the document is flagged with what is still missing.
#   cut          pieces as (start, end, label): the gap before the first heading is the
#                opening piece; pieces the model placed after the end matter are dropped.


def locate_all(text, reply):
    at, missing = {}, []

    def place(key, quote, start):
        hit = find(text, quote, start)
        if hit is None:
            missing.append(quote)
        else:
            at[key] = hit
        return hit

    pos = 0
    if reply.get("contents_end"):
        hit = place("contents_end", reply["contents_end"], 0)
        pos = hit + len(reply["contents_end"]) if hit is not None else 0
    if reply.get("body_start"):
        hit = place("body_start", reply["body_start"], pos)
        pos = hit if hit is not None else pos
    for i, piece in enumerate(reply["pieces"]):
        hit = place(f"piece {i}", piece.get("marker"), pos)
        pos = hit + 1 if hit is not None else pos
    if reply.get("end_matter_start"):
        place("end_matter_start", reply["end_matter_start"], at.get("body_start", 0))
    for quote in (reply.get("title"), reply.get("author"), (reply.get("date") or {}).get("quote")):
        if quote and find(text, quote) is None:
            missing.append(quote)
    return at, missing


def until_found(doc, attempts=2):
    feedback = ""
    for i in range(attempts):
        reply = propose(doc, feedback=feedback)
        at, missing = locate_all(doc["text"], reply)
        if not missing:
            return reply, at, missing, i + 1
        feedback = ("\nThese strings from a previous answer were not found in the text. Copy every string "
                    "exactly as it appears: " + json.dumps(missing[:20], ensure_ascii=False) + "\n")
    return reply, at, missing, attempts


def cut(text, reply, at):
    end = at.get("end_matter_start", len(text))
    starts = [(at[f"piece {i}"], p.get("title") or p.get("marker"))
              for i, p in enumerate(reply["pieces"]) if f"piece {i}" in at]
    kept = [(s, label) for s, label in starts if s < end]
    body_start = at.get("body_start", kept[0][0] if kept else 0)
    pieces = []
    if kept and text[body_start:kept[0][0]].strip():
        pieces.append((body_start, kept[0][0], "opening"))
    bounds = [s for s, _ in kept] + [end]
    pieces += [(s, e, label) for (s, label), e in zip(kept, bounds[1:])]
    if not kept:
        pieces.append((body_start, end, reply.get("title") or "whole"))
    return pieces, len(starts) - len(kept)


In [ ]:
# Block 7: every text and PDF in the dataset, resumable. Each finished document is appended
# to splits.jsonl (reply, positions, pieces, cost) and skipped on a rerun. Chats need no
# model call and are handled at export. One line per document; details for flagged ones.
SPLITS = Path("/kaggle/working/splits.jsonl")
done = set()
if SPLITS.exists():
    done = {json.loads(line)["sha256"] for line in SPLITS.read_text(encoding="utf-8").splitlines()}

paths = sorted(RAW.glob("oz/*.txt")) + sorted(RAW.glob("holmes/*.txt")) + sorted(RAW.glob("greek/*.txt")) \
      + sorted(RAW.glob("graphrag-bench/*.txt")) + sorted(PAPERS.glob("*.pdf"))

for path in paths:
    doc = to_text(path)
    if doc["sha256"] in done:
        continue
    before = spend()
    reply, at, missing, tries = until_found(doc)
    pieces, dropped = cut(doc["text"], reply, at)
    record = {"path": str(path), "sha256": doc["sha256"], "reply": reply, "at": at, "missing": missing,
              "tries": tries, "pieces": pieces, "dropped": dropped, "cost": round(spend() - before, 4)}
    with SPLITS.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    done.add(doc["sha256"])
    words = [len(doc["text"][s:e].split()) for s, e, _ in pieces]
    flag = "FLAGGED" if missing else "ok"
    print(f"{path.name:<34} {flag:<7} pieces {len(pieces):>3}  dropped {dropped:>2}  tries {tries}  toc {reply.get('toc_count')!s:>4}"
          f"  words min {min(words):>5} max {max(words):>6}  date {(reply.get('date') or {}).get('iso') or '-':<10}  ${record['cost']:.3f}")
    for quote in missing[:3]:
        print(f"      MISSING {quote!r}")
print(f"\n{len(done)} documents in {SPLITS.name}, ${spend():.2f} spent this session")
